# ArduPilot SITL ↔ Isaac Sim — lockstep-мост

**Порядок: прогнать ячейки СВЕРХУ ВНИЗ до «Диагностики» включительно, потом WSL.**

**Один раз на систему** (PowerShell от админа):
```powershell
New-NetFirewallRule -DisplayName "ArduPilot JSON UDP 9002" -Direction Inbound -Protocol UDP -LocalPort 9002 -Action Allow
```

**Терминал 1 — SITL** (wsl_bridge.py больше НЕ используется):
```bash
cd ~/ardupilot
WIN_IP=$(grep nameserver /etc/resolv.conf | awk '{print $2}')
python3 Tools/autotest/sim_vehicle.py -v ArduCopter --model JSON:$WIN_IP --add-param-file=/mnt/c/VSCODE/ISAAC/sitl_defaults.parm --no-rebuild
```
В MavProxy: `output add 127.0.0.1:14550`

**Проверка**: ячейка «Диагностика» дважды — «Пакетов» растёт, `dt_ms ≈ 4.17` (физика 240Hz).

**Терминал 2 — ARM**:
```bash
python3 /mnt/c/VSCODE/ISAAC/arm_takeoff.py
```

**После краша**: Ctrl+C в SITL → Stop→Play в Isaac → ячейка «Сброс» → заново SITL → arm_takeoff.

**Фикс-11 (Iris)** — в самом низу, выполнять ТОЛЬКО если тест Фикс-10 не убрал раскачку.

## 1. Связь с Isaac Sim (порт 8226, extension python_server)

In [ ]:
import asyncio
import json

async def execute_in_isaac(source: str, host: str = "127.0.0.1", port: int = 8226) -> dict:
    """Отправить код в открытый Isaac Sim и получить результат."""
    reader, writer = await asyncio.open_connection(host, port)
    writer.write(source.encode())
    writer.write_eof()
    data = await reader.read()
    writer.close()
    return json.loads(data.decode())

In [ ]:
result = await execute_in_isaac('''
import omni.usd
stage = omni.usd.get_context().get_stage()
print("Связь с Isaac: OK")
print("Дрон /World/cf2x существует:", stage.GetPrimAtPath("/World/cf2x").IsValid())
''')
print(result.get("output",""), result.get("traceback",""))

## 2. Снять Drive с пропеллеров (суставы свободны, визуальное вращение не мешает физике)

In [ ]:
result = await execute_in_isaac('''
import omni.usd
from pxr import UsdPhysics

stage = omni.usd.get_context().get_stage()
joint_paths = [
    "/World/cf2x/body/m1_joint",
    "/World/cf2x/body/m2_joint",
    "/World/cf2x/body/m3_joint",
    "/World/cf2x/body/m4_joint",
]
for path in joint_paths:
    prim = stage.GetPrimAtPath(path)
    drive = UsdPhysics.DriveAPI.Apply(prim, "angular")
    drive.CreateTypeAttr("force")
    drive.CreateTargetVelocityAttr(0.0)
    drive.CreateDampingAttr(0.0)
    drive.CreateStiffnessAttr(0.0)
print("Drive снят — суставы пропеллеров свободны")
''')
print(result["status"], result.get("output"), result.get("traceback", ""))

## 3. Конфигурация дрона (масса, физический контекст, глобалы stage/sim_iface/BODY_ID)

In [ ]:
result = await execute_in_isaac('''
DRONE_ROOT = "/World/cf2x"
PROP_PATHS = [
    f"{DRONE_ROOT}/m1_prop",
    f"{DRONE_ROOT}/m2_prop",
    f"{DRONE_ROOT}/m3_prop",
    f"{DRONE_ROOT}/m4_prop",
]
BODY_PATH = f"{DRONE_ROOT}/body"

# Масса дрона (кг). None = взять из симуляции (PhysicsMassAPI). Фикс-11 переопределяет.
MASS_OVERRIDE_KG = None
GRAVITY = 9.81
# Запас тяги над весом (макс тяга мотора = вес/4 * headroom). Фикс-11 переопределяет константы напрямую.
THRUST_HEADROOM = 6.0

import omni.physx
import omni.usd
from pxr import PhysicsSchemaTools, Sdf, UsdGeom, Usd, UsdPhysics
import carb

stage = omni.usd.get_context().get_stage()
stage_id = omni.usd.get_context().get_stage_id()
sim_iface = omni.physx.get_physx_simulation_interface()
physx_iface = omni.physx.get_physx_interface()
BODY_ID = PhysicsSchemaTools.sdfPathToInt(Sdf.Path(BODY_PATH))

def _get_total_mass():
    if MASS_OVERRIDE_KG is not None:
        return MASS_OVERRIDE_KG
    total = 0.0
    for p in [BODY_PATH] + PROP_PATHS:
        prim = stage.GetPrimAtPath(p)
        m = UsdPhysics.MassAPI(prim).GetMassAttr().Get()
        total += m if m else 0.0
    return total

DRONE_MASS_KG = _get_total_mass()
HOVER_FORCE_TOTAL_N = DRONE_MASS_KG * GRAVITY
HOVER_FORCE_PER_MOTOR_N = HOVER_FORCE_TOTAL_N / 4
MAX_FORCE_PER_MOTOR_N = HOVER_FORCE_PER_MOTOR_N * THRUST_HEADROOM

print(f"Масса дрона: {DRONE_MASS_KG:.4f} кг")
print(f"Тяга для зависания (всего): {HOVER_FORCE_TOTAL_N:.4f} Н")
print(f"Макс. тяга на мотор: {MAX_FORCE_PER_MOTOR_N:.4f} Н")
''')
print(result["status"])
print(result.get("output"))
print(result.get("traceback", ""))

## 4. Геометрия моторов + read_state() — выполнять пока дрон в исходной позе (до Play/краша)

In [ ]:
result = await execute_in_isaac('''
import numpy as np
from scipy.spatial.transform import Rotation
from pxr import Gf, UsdPhysics

MAX_ROTOR_VELOCITY = 1100.0

PROP_WORLD_POS = [
    UsdGeom.Xformable(stage.GetPrimAtPath(path)).ComputeLocalToWorldTransform(Usd.TimeCode.Default()).ExtractTranslation()
    for path in PROP_PATHS
]
PROP_CENTROID_XY = (
    sum(p[0] for p in PROP_WORLD_POS) / 4,
    sum(p[1] for p in PROP_WORLD_POS) / 4,
)
PROP_OFFSETS_XY = [(p[0] - PROP_CENTROID_XY[0], p[1] - PROP_CENTROID_XY[1]) for p in PROP_WORLD_POS]
ROT_DIR = [+1, -1, +1, -1]
ROTOR_CONSTANT = MAX_FORCE_PER_MOTOR_N / (MAX_ROTOR_VELOCITY ** 2)
ROLLING_MOMENT_COEFF = ROTOR_CONSTANT * 0.117
ROTOR_CONSTANTS = [ROTOR_CONSTANT] * 4
ROLLING_MOMENT_COEFFS = [ROLLING_MOMENT_COEFF] * 4

def read_state():
    """Позиция, ориентация, скорость и угловая скорость тела дрона."""
    body_prim = stage.GetPrimAtPath(BODY_PATH)
    transform = UsdGeom.Xformable(body_prim).ComputeLocalToWorldTransform(Usd.TimeCode.Default())
    pos = transform.ExtractTranslation()
    quat = transform.ExtractRotationQuat()
    imag = quat.GetImaginary()
    R = Rotation.from_quat([imag[0], imag[1], imag[2], quat.GetReal()])
    rb = UsdPhysics.RigidBodyAPI(body_prim)
    v = rb.GetVelocityAttr().Get()
    w_deg = rb.GetAngularVelocityAttr().Get()
    w = np.radians([w_deg[0], w_deg[1], w_deg[2]])
    return np.array([pos[0], pos[1], pos[2]]), R, np.array([v[0], v[1], v[2]]), w

print("Смещения моторов:", PROP_OFFSETS_XY)
print(f"ROTOR_CONSTANT: {ROTOR_CONSTANT:.4e}")
print("read_state() готова")
''')
print(result["status"])
print(result.get("output", ""))
print(result.get("traceback", ""))

## 5. Фикс-10 — физика 240Hz (тест: убрать раскачку rate-контура)

PID ArduPilot не справлялись при контуре 50Hz и физике 60Hz (задержка ~35мс). Пара к этой ячейке: `SCHED_LOOP_RATE 200` уже стоит в `sitl_defaults.parm` и `arm_takeoff.py`. Ячейка сама находит физическую сцену по типу (или создаёт). Выполнять ДО Play.

In [ ]:
result = await execute_in_isaac('''
from pxr import UsdPhysics, PhysxSchema, Sdf

_scene_prim = None
for _p in stage.Traverse():
    if _p.IsA(UsdPhysics.Scene):
        _scene_prim = _p
        break

if _scene_prim is None:
    _scene = UsdPhysics.Scene.Define(stage, Sdf.Path("/World/PhysicsScene"))
    _scene_prim = _scene.GetPrim()
    print("PhysicsScene не было — создан", _scene_prim.GetPath())

_ps_api = PhysxSchema.PhysxSceneAPI.Apply(_scene_prim)
_ps_api.CreateTimeStepsPerSecondAttr().Set(240)
print("PhysicsScene:", _scene_prim.GetPath())
print("timeStepsPerSecond =", _ps_api.GetTimeStepsPerSecondAttr().Get())
''')
print(result.get("output",""))
print(result.get("traceback",""))

## 6. Play

In [ ]:
result = await execute_in_isaac('''
import omni.timeline
omni.timeline.get_timeline_interface().play()
print("Симуляция запущена")
''')
print(result["status"], result.get("output"))

## 7. Инфраструктура моста (общий стейт; ставит `_wsl_active=True`; убирает legacy авто-сброс)

In [ ]:
result = await execute_in_isaac('''
import threading
import numpy as np

_MAX_VEL = globals().get("MAX_ROTOR_VELOCITY", 1100.0)
def _pwm_to_vel(pwm):
    return max(0.0, (max(1000, min(2000, pwm)) - 1000) / 1000.0 * _MAX_VEL)

_missing = [g for g in ["ROTOR_CONSTANTS","PROP_OFFSETS_XY","ROT_DIR","ROLLING_MOMENT_COEFFS",
                         "read_state","stage","sim_iface","physx_iface","BODY_PATH","BODY_ID"]
            if g not in globals()]
if _missing:
    raise RuntimeError("Не определены: " + str(_missing) + ". Запусти ячейки выше!")

_wsl_servos = [1000] * 16
_wsl_state  = {
    "imu":        {"gyro": [0, 0, 0], "accel_body": [0, 0, -9.81]},
    "position":   [0.0, 0.0, 0.0],
    "velocity":   [0.0, 0.0, 0.0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
}
_wsl_t_ms   = 0.0
_wsl_lock   = threading.Lock()
_wsl_active = True

# Убираем legacy авто-сброс из старых сессий (он ставил _wsl_active=False на каждый Stop)
try:
    _timeline_sub.unsubscribe()
except Exception:
    pass

print("Инфраструктура OK, _wsl_active =", _wsl_active)
print("Дальше — ячейка Фикс-9")
''')
print(result["status"])
print(result.get("output",""))
print(result.get("traceback",""))

## 8. Фикс-12 — ГЛАВНЫЙ КОЛБЭК (lockstep + ИСПРАВЛЕННЫЕ ФРЕЙМЫ по тесту знаков)

Тест знаков (2026-07-08) показал: реальный мир сцены — `X=North, Y=WEST, Z=Up`, а не Y=East. Отсюда две ошибки прошлых версий: SERVO_REMAP был зеркален лево-право (команда «правый бок вниз» валила влево), и все Y-компоненты сенсоров шли без смены знака — кватернион противоречил гироскопу → вечные EKF variance → «волчок».

**Что здесь**: lockstep UDP :9002 + правильная конверсия FLU→FRD (Y-знак меняется у gyro/accel/position/velocity/quaternion) + исправленный `SERVO_REMAP=[0,3,1,2]` и `ROT_DIR=[-1,+1,-1,+1]` + accel Пегаса + броня + watchdog. Если перезапускал ячейку 4 (геометрия) — перезапусти эту (она переопределяет ROT_DIR).

In [ ]:
code = '''
import socket, struct, json, math
import numpy as np
from scipy.spatial.transform import Rotation as _R

# === Фикс-12: исправленный motor-map (лево-право было зеркально) ===
SERVO_REMAP = [0, 3, 1, 2]   # rotor i <- AP servo[SERVO_REMAP[i]]; роторы: FR, RR, RL, FL
ROT_DIR = [-1, +1, -1, +1]   # FR=CCW, RR=CW, RL=CCW, FL=CW (переопределяет геометрию)
AP_JSON_PORT = 9002
_SERVO_MAGIC = 18458  # 0x47FA
_WATCHDOG_MS = 1000.0
_GRAVITY_WORLD = np.array([0.0, 0.0, -9.81])

def _clamp(v, lo, hi):
    return max(lo, min(hi, float(v)))

# --- UDP-сокет JSON-протокола (идемпотентно) ---
try:
    _ap_udp_sock.close()
except Exception:
    pass
_ap_udp_sock = socket.socket(socket.AF_INET, socket.SOCK_DGRAM)
_ap_udp_sock.setsockopt(socket.SOL_SOCKET, socket.SO_REUSEADDR, 1)
_ap_udp_sock.bind(("0.0.0.0", AP_JSON_PORT))
_ap_udp_sock.setblocking(False)
try:
    _ap_udp_sock.ioctl(-1744830452, False)  # SIO_UDP_CONNRESET (Windows-квирк)
except Exception:
    pass

_ap_addr = None
_ap_pkt_count = 0
_ap_reply_count = 0
_ap_last_pkt_t_ms = -1e9
_ap_watchdog_fired = False
_prev_v_world = np.array([0.0, 0.0, 0.0])
_cb_error_count = 0
_cb_last_error = ""
_last_good_state = {
    "imu":        {"gyro": [0.0, 0.0, 0.0], "accel_body": [0.0, 0.0, -9.81]},
    "position":   [0.0, 0.0, 0.0],
    "velocity":   [0.0, 0.0, 0.0],
    "quaternion": [1.0, 0.0, 0.0, 0.0],
}
_wsl_log = []
_wsl_log_last_t = -1e9
_LOG_INTERVAL_MS = 50.0
_LOG_MAX_ENTRIES = 6000

def _parse_servos(data):
    if len(data) >= 40 and struct.unpack_from("<H", data, 0)[0] == _SERVO_MAGIC:
        return list(struct.unpack_from("<16H", data, 8))
    if len(data) >= 36 and struct.unpack_from("<H", data, 0)[0] == _SERVO_MAGIC:
        return list(struct.unpack_from("<16H", data, 4))
    return None

def _on_physics_step_wsl(dt):
    global _wsl_state, _wsl_t_ms, _ap_addr, _ap_pkt_count, _ap_reply_count
    global _ap_last_pkt_t_ms, _ap_watchdog_fired, _wsl_log_last_t
    global _prev_v_world, _cb_error_count, _cb_last_error, _last_good_state
    if dt <= 0 or not _wsl_active: return
    _wsl_t_ms += dt * 1000.0

    # 1. Servo-пакеты (non-blocking drain)
    got_packet = False
    for _i in range(64):
        try:
            data, addr = _ap_udp_sock.recvfrom(4096)
        except (BlockingIOError, OSError):
            break
        s = _parse_servos(data)
        if s is not None:
            _wsl_servos[:] = s
            _ap_addr = addr
            got_packet = True
            _ap_pkt_count += 1
    if got_packet:
        _ap_last_pkt_t_ms = _wsl_t_ms
        _ap_watchdog_fired = False

    # 1b. WATCHDOG
    if (_ap_pkt_count > 0 and not _ap_watchdog_fired
            and _wsl_t_ms - _ap_last_pkt_t_ms > _WATCHDOG_MS):
        _wsl_servos[:] = [1000] * 16
        _ap_watchdog_fired = True
        print("[WATCHDOG] SITL молчит > " + str(int(_WATCHDOG_MS)) + "мс — моторы выключены")

    servos = list(_wsl_servos)

    # 2+3. Физика и сенсоры в try/except: при ошибке реюз последнего валидного state
    try:
        mv = [_pwm_to_vel(servos[SERVO_REMAP[i]]) for i in range(4)]
        fn = [ROTOR_CONSTANTS[i] * mv[i]**2 for i in range(4)]
        total_thrust = sum(fn)
        tau_roll  = sum( PROP_OFFSETS_XY[i][1] * fn[i] for i in range(4))
        tau_pitch = sum(-PROP_OFFSETS_XY[i][0] * fn[i] for i in range(4))
        tau_yaw   = sum(ROLLING_MOMENT_COEFFS[i] * mv[i]**2 * ROT_DIR[i] for i in range(4))
        body_prim = stage.GetPrimAtPath(BODY_PATH)
        xf = UsdGeom.Xformable(body_prim).ComputeLocalToWorldTransform(Usd.TimeCode.Default())
        up = xf.TransformDir(Gf.Vec3d(0, 0, 1)).GetNormalized()
        bp = xf.ExtractTranslation()
        sim_iface.apply_force_at_pos(
            stage_id, BODY_ID,
            carb.Float3(float(up[0]*total_thrust), float(up[1]*total_thrust), float(up[2]*total_thrust)),
            carb.Float3(float(bp[0]), float(bp[1]), float(bp[2])), "Force")
        q_usd = xf.ExtractRotationQuat()
        im = q_usd.GetImaginary()
        q_sc = _R.from_quat([im[0], im[1], im[2], q_usd.GetReal()])
        Rm = q_sc.as_matrix()
        tw = Rm @ np.array([tau_roll, tau_pitch, tau_yaw])
        sim_iface.apply_torque(stage_id, BODY_ID, carb.Float3(float(tw[0]), float(tw[1]), float(tw[2])))

        _, R, v, w_world = read_state()
        # Гироскоп мир -> тело
        w = Rm.T @ np.array([w_world[0], w_world[1], w_world[2]])

        # accel Пегаса: удельная сила = (реальное ускорение - гравитация)
        a_true_world = (v - _prev_v_world) / dt
        _prev_v_world = v.copy()
        f_world = a_true_world - _GRAVITY_WORLD
        f_body = Rm.T @ f_world

        # === Фикс-12: конверсия FLU->FRD (мир Isaac: X=North, Y=West, Z=Up) ===
        # Y-компонента меняет знак у ВСЕХ величин; кватернион: (w, x, -y, -z)
        qn = q_sc.as_quat()
        new_state = {
            "imu": {
                "gyro":       [_clamp(w[0],-50,50),        _clamp(-w[1],-50,50),       _clamp(-w[2],-50,50)],
                "accel_body": [_clamp(f_body[0],-50,50),   _clamp(-f_body[1],-50,50),  _clamp(-f_body[2],-50,50)],
            },
            "position":   [_clamp(bp[0],-1e4,1e4),  _clamp(-bp[1],-1e4,1e4),  _clamp(-bp[2],-1e4,1e4)],
            "velocity":   [_clamp(v[0],-200,200),    _clamp(-v[1],-200,200),    _clamp(-v[2],-200,200)],
            "quaternion": [float(qn[3]), float(qn[0]), float(-qn[1]), float(-qn[2])],
        }
        _vals = (new_state["position"] + new_state["velocity"] + new_state["quaternion"]
                 + new_state["imu"]["gyro"] + new_state["imu"]["accel_body"])
        if not all(math.isfinite(x) for x in _vals):
            raise ValueError("non-finite value in state")
        _last_good_state = new_state
    except Exception as e:
        _cb_error_count += 1
        _cb_last_error = repr(e)
        new_state = _last_good_state
        total_thrust = 0.0
        up = None

    with _wsl_lock:
        _wsl_state = new_state

    # 4. LOCKSTEP-ответ — уходит ВСЕГДА при got_packet
    if got_packet and _ap_addr is not None:
        _rc3 = 1500 if any(p > 1000 for p in servos[:4]) else 1100
        resp = {
            "timestamp":  _wsl_t_ms / 1000.0,
            "imu":        new_state["imu"],
            "position":   new_state["position"],
            "velocity":   new_state["velocity"],
            "quaternion": new_state["quaternion"],
            "rc": {"rc_1":1500,"rc_2":1500,"rc_3":_rc3,"rc_4":1500,
                   "rc_5":1500,"rc_6":1500,"rc_7":1500,"rc_8":1500},
        }
        try:
            _ap_udp_sock.sendto((json.dumps(resp, separators=(",", ":")) + chr(10)).encode(), _ap_addr)
            _ap_reply_count += 1
        except Exception:
            pass

    # 5. Лог ~20Hz (пропускается на шагах с ошибкой)
    if (up is not None and _wsl_t_ms - _wsl_log_last_t >= _LOG_INTERVAL_MS
            and len(_wsl_log) < _LOG_MAX_ENTRIES):
        _wsl_log_last_t = _wsl_t_ms
        tilt_deg = float(np.degrees(np.arccos(max(-1.0, min(1.0, up[2])))))
        _wsl_log.append({
            "t":        round(_wsl_t_ms / 1000.0, 2),
            "dt_ms":    round(dt * 1000.0, 2),
            "alt":      round(-bp[2], 3),
            "x":        round(bp[0], 3),
            "y":        round(bp[1], 3),
            "thrust":   round(total_thrust, 3),
            "accel_z":  round(float(new_state["imu"]["accel_body"][2]), 2),
            "tilt_deg": round(tilt_deg, 1),
            "gyro_deg": [round(float(np.degrees(x)), 1) for x in w],
            "pwm":      servos[:4],
        })

try: _ap_subscription.unsubscribe()
except: pass
_ap_subscription = physx_iface.subscribe_physics_step_events(_on_physics_step_wsl)
print("OK: Фикс-12 активирован — FLU->FRD конверсии + исправленный SERVO_REMAP/ROT_DIR")
'''
result = await execute_in_isaac(code)
print(result["status"])
print(result.get("output",""))
print(result.get("traceback",""))

## 9. Диагностика + лог полёта (запускать сколько угодно раз)

До ARM: «Пакетов» должно расти между запусками ячейки, `dt_ms ≈ 4.17` (240Hz). После теста: полётная часть лога (PWM > 1000).

In [ ]:
result = await execute_in_isaac('''
print("_wsl_active =", _wsl_active)
print("Пакетов от ArduPilot:", _ap_pkt_count, "  Ответов Isaac:", _ap_reply_count)
print("Адрес SITL:", _ap_addr)
_last = globals().get("_ap_last_pkt_t_ms", None)
if _last is not None and _ap_pkt_count > 0:
    print("Возраст последнего пакета: {:.2f} сек сим-времени".format((_wsl_t_ms - _last) / 1000.0))
    print("Watchdog сработал:", globals().get("_ap_watchdog_fired", "-"))
_errs = globals().get("_cb_error_count", None)
if _errs is not None:
    print("Ошибок колбэка:", _errs, " Последняя:", globals().get("_cb_last_error", ""))
print("Всего записей лога:", len(_wsl_log))
flight = [e for e in _wsl_log if any(p > 1000 for p in e["pwm"])]
print("Полётных записей (PWM>1000):", len(flight))
lines = []
for e in flight:
    lines.append(
        "t={:7.2f}s dt={:5.2f}ms alt={:+6.2f} x={:+5.2f} y={:+5.2f} thr={:.3f}N az={:+6.2f} tilt={:5.1f}deg gyro={} pwm={}".format(
            e["t"], e["dt_ms"], e["alt"], e["x"], e["y"], e["thrust"], e.get("accel_z", 0.0),
            e["tilt_deg"], e["gyro_deg"], e["pwm"]
        )
    )
print(chr(10).join(lines))
''')
print(result.get("output",""))
print(result.get("traceback",""))

## Сброс (после Stop→Play / перед новым тестом): моторы в ноль, мост включён

In [ ]:
result = await execute_in_isaac(
    '_wsl_servos[:] = [1000] * 16; _wsl_active = True; print("servos=1000, _wsl_active=True")'
)
print(result.get("output",""))

## Тест знаков (открытый цикл, БЕЗ ArduPilot!)

Диагностика «волчка»: вручную подаём чистые PWM-паттерны yaw/roll/pitch (как их подал бы ArduPilot) и измеряем, куда реально вращается дрон. Никаких догадок — только измеренные знаки.

**Перед запуском: SITL должен быть ВЫКЛЮЧЕН (Ctrl+C)** — иначе его пакеты перезапишут ручные серво. Требует прогнанных ячеек до Фикс-9 включительно. Ячейка сама делает Stop→Play между пробами и в конце. Дрон будет подпрыгивать и вращаться — это и есть тест. Пришли весь вывод.

In [ ]:
import asyncio

async def _srv(m1, m2, m3, m4):
    await execute_in_isaac("_wsl_servos[:] = [%d,%d,%d,%d] + [1000]*12" % (m1, m2, m3, m4))

async def _tl(cmd):
    await execute_in_isaac("import omni.timeline as _t; _t.get_timeline_interface().%s()" % cmd)

# Защита: SITL должен быть выключен (иначе его пакеты перезапишут серво)
r1 = await execute_in_isaac("print(_ap_pkt_count)")
await asyncio.sleep(1.0)
r2 = await execute_in_isaac("print(_ap_pkt_count)")
if r1.get("output") != r2.get("output"):
    print("СТОП: SITL работает (пакеты идут). Выключи его (Ctrl+C) и запусти тест снова.")
else:
    # Паттерны в порядке AP-моторов M1..M4 (_wsl_servos[0..3]).
    # QUAD/X: M1=перед-право(CCW), M2=зад-лево(CCW), M3=перед-лево(CW), M4=зад-право(CW)
    tests = [
        ("YAW+   (верх M1,M2 - AP командует 'yaw вправо, по часовой сверху')", (1700, 1700, 1300, 1300)),
        ("ROLL+  (верх M2,M3 - AP командует 'правый бок вниз')",               (1300, 1700, 1700, 1300)),
        ("PITCH+ (верх M1,M3 - AP командует 'нос вверх')",                     (1700, 1300, 1700, 1300)),
    ]
    print("Тест знаков: 3 пробы, каждая со сбросом сцены...")
    for name, pat in tests:
        await _srv(1000, 1000, 1000, 1000)
        await _tl("stop")
        await asyncio.sleep(0.7)
        await _tl("play")
        await asyncio.sleep(0.7)
        await _srv(1500, 1500, 1500, 1500)   # отрыв (тяга ~1.5x веса)
        await asyncio.sleep(0.35)
        await _srv(*pat)                     # импульс паттерна
        await asyncio.sleep(0.3)
        r = await execute_in_isaac(
            "print(_wsl_log[-1]['gyro_deg'], '| tilt', _wsl_log[-1]['tilt_deg'], 'deg | alt', _wsl_log[-1]['alt'])"
        )
        await _srv(1000, 1000, 1000, 1000)
        print()
        print(name)
        print("   gyro тела [x,y,z] град/с (фрейм z-up):", r.get("output", "").strip())
    await _tl("stop")
    await asyncio.sleep(0.5)
    await _tl("play")
    await _srv(1000, 1000, 1000, 1000)
    print()
    print("Готово. Пришли весь вывод — по знакам вычислим правильные конверсии.")

## Фикс-11 (ЗАПАСНОЙ) — репспек дрона в Iris (проверенная пара из Pegasus)

**Выполнять ТОЛЬКО если тест Фикс-10 не убрал раскачку при отрыве.**

Pegasus летает на Iris (1.5кг) с константами из `quadratic_thrust_curve.py` и SITL-моделью `gazebo-iris`, у которой в ArduPilot готовый PID-тюн. Берём пару целиком: дрон физически становится Iris (масса, инерция, плечи, константы тяги), SITL получает Iris-тюн.

**Применение**: ячейка ниже → Stop→Play → ячейка «Сброс» → SITL с ДВУМЯ parm (наш — ПОСЛЕДНИМ, чтобы его ARMING_CHECK/SCHED_LOOP_RATE победили):
```bash
cd ~/ardupilot
WIN_IP=$(grep nameserver /etc/resolv.conf | awk '{print $2}')
python3 Tools/autotest/sim_vehicle.py -v ArduCopter --model JSON:$WIN_IP \
  --add-param-file=$HOME/ardupilot/Tools/autotest/default_params/gazebo-iris.parm \
  --add-param-file=/mnt/c/VSCODE/ISAAC/sitl_defaults.parm --no-rebuild
```

**ВАЖНО**: после Фикс-11 НЕ перезапускать ячейки 3-4 (конфиг/геометрия) — затрут константы. Если перезапустил — выполни Фикс-11 снова. Колбэк Фикс-9 перезапускать не нужно.

In [ ]:
code = '''
import numpy as np
from pxr import UsdPhysics, Gf

# === Спецификация Iris — точные числа из Pegasus (quadratic_thrust_curve.py) и iris.sdf ===
IRIS_MASS_KG        = 1.5
IRIS_INERTIA        = (0.029125, 0.029125, 0.055225)  # кг*м^2 (Ixx, Iyy, Izz)
IRIS_ARM_X          = 0.13
IRIS_ARM_Y          = 0.21
IRIS_ROTOR_CONSTANT = 8.54858e-6
IRIS_ROLLING_COEFF  = 1e-6

body_prim = stage.GetPrimAtPath(BODY_PATH)
_mass_api = UsdPhysics.MassAPI.Apply(body_prim)
_mass_api.CreateMassAttr().Set(IRIS_MASS_KG)
_mass_api.CreateDiagonalInertiaAttr().Set(Gf.Vec3f(*IRIS_INERTIA))
_mass_api.CreatePrincipalAxesAttr().Set(Gf.Quatf(1.0, 0.0, 0.0, 0.0))

DRONE_MASS_KG = IRIS_MASS_KG
MASS_OVERRIDE_KG = IRIS_MASS_KG
HOVER_FORCE_TOTAL_N = DRONE_MASS_KG * GRAVITY
MAX_FORCE_PER_MOTOR_N = IRIS_ROTOR_CONSTANT * (MAX_ROTOR_VELOCITY ** 2)
ROTOR_CONSTANT = IRIS_ROTOR_CONSTANT
ROLLING_MOMENT_COEFF = IRIS_ROLLING_COEFF
ROTOR_CONSTANTS = [ROTOR_CONSTANT] * 4
ROLLING_MOMENT_COEFFS = [ROLLING_MOMENT_COEFF] * 4
# Порядок роторов наш: FL, RL, RR, FR (знаковый паттерн сохранён, плечи Iris)
PROP_OFFSETS_XY = [( IRIS_ARM_X, -IRIS_ARM_Y),
                   (-IRIS_ARM_X, -IRIS_ARM_Y),
                   (-IRIS_ARM_X,  IRIS_ARM_Y),
                   ( IRIS_ARM_X,  IRIS_ARM_Y)]

print("Масса body:", _mass_api.GetMassAttr().Get(), "кг")
print("Инерция (Ixx,Iyy,Izz):", _mass_api.GetDiagonalInertiaAttr().Get())
print("Плечи: x=+-{} y=+-{} м".format(IRIS_ARM_X, IRIS_ARM_Y))
print("Вес: {:.1f}Н  Макс тяга (4 мотора): {:.1f}Н  Запас: {:.2f}x".format(
    HOVER_FORCE_TOTAL_N, 4*MAX_FORCE_PER_MOTOR_N, 4*MAX_FORCE_PER_MOTOR_N/HOVER_FORCE_TOTAL_N))
print("Дальше: Stop -> Play -> ячейка Сброс -> SITL с gazebo-iris.parm -> arm_takeoff")
'''
result = await execute_in_isaac(code)
print(result["status"])
print(result.get("output",""))
print(result.get("traceback",""))